# Binary Classification of Government-Related Funding Organizations

This notebook compares BERT and SciBERT for binary classification of organization names.

- Class 0: Non-government-related funding organization
- Class 1: Government-related funding organization
- Evaluation method: Stratified 10-fold cross-validation
- Input feature: `Org_Name`
- Target variable: `Class`

In [1]:
# Install only the packages required for text classification.
%pip install -q --upgrade \
    transformers \
    datasets \
    accelerate \
    scikit-learn \
    pandas \
    openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import torch

print(f"Python executable: {sys.executable}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Python executable: /home/ec2-user/anaconda3/envs/pytorch/bin/python
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: NVIDIA A10G


## 1. Import Libraries

In [3]:
import gc
import json
import os
import random
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)
from sklearn.model_selection import StratifiedKFold
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)

## 2. Experiment Configuration

In [4]:
# =============================================================================
# File and directory settings
# =============================================================================

DATA_FILE = "org_name_BC.csv"

OUTPUT_ROOT = Path("./organization_classification_results")
CV_OUTPUT_DIR = OUTPUT_ROOT / "cross_validation"
FINAL_MODEL_DIR = OUTPUT_ROOT / "final_model"
RESULTS_DIR = OUTPUT_ROOT / "evaluation_results"

# =============================================================================
# Model selection
# =============================================================================

# Select either "bert" or "scibert"
MODEL_CHOICE = "bert"

MODEL_REGISTRY = {
    # Multilingual BERT is recommended because the organization names
    # contain multiple languages and non-English characters.
    "bert": "google-bert/bert-base-multilingual-cased",

    # SciBERT was pretrained primarily on English scientific publications.
    "scibert": "allenai/scibert_scivocab_uncased"
}

# Validate the selected model
if MODEL_CHOICE not in MODEL_REGISTRY:
    raise ValueError(
        f"MODEL_CHOICE must be one of {list(MODEL_REGISTRY.keys())}."
    )

MODEL_NAME = MODEL_REGISTRY[MODEL_CHOICE]

# =============================================================================
# Reproducibility settings
# =============================================================================

RANDOM_SEED = 42
N_SPLITS = 10

# =============================================================================
# Tokenization and training settings
# =============================================================================

# Organization names are generally short, so 64 tokens should be sufficient.
# This value can be increased if the dataset contains unusually long names.
MAX_LENGTH = 64

LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
NUM_EPOCHS = 5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRADIENT_ACCUMULATION_STEPS = 1

# Delete fold checkpoints after metrics and predictions are collected.
# Set to True only when fold-level model files must be retained.
KEEP_FOLD_CHECKPOINTS = False

# Use mixed-precision training when a CUDA GPU is available.
USE_FP16 = torch.cuda.is_available()

# =============================================================================
# Column settings
# =============================================================================

TEXT_COLUMN = "Org_Name"
LABEL_COLUMN = "Class"
ID_COLUMN = "Instance_ID"

# =============================================================================
# Label definitions
# =============================================================================

ID2LABEL = {
    0: "non-gov",
    1: "gov"
}

LABEL2ID = {
    "non-gov": 0,
    "gov": 1
}

# =============================================================================
# Create output directories
# =============================================================================

CV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Selected model type: {MODEL_CHOICE}")
print(f"Hugging Face model: {MODEL_NAME}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Selected model type: bert
Hugging Face model: google-bert/bert-base-multilingual-cased
CUDA available: True
GPU: NVIDIA A10G


## 3. Set Random Seeds

In [5]:
def set_seed(seed: int) -> None:
    """Set random seeds for reproducible experiments."""

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # These settings improve reproducibility but may reduce training speed.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(RANDOM_SEED)

## 4. Load and Validate the Dataset

In [6]:
# =============================================================================
# Load the UTF-8 Dataset
# =============================================================================


# Load the CSV file using UTF-8 with BOM support.
# UTF-8-SIG also reads ordinary UTF-8 files correctly.
df = pd.read_csv(
    DATA_FILE,
    encoding="utf-8-sig"
)

print(f"Dataset shape: {df.shape}")
display(df.head())

Dataset shape: (500, 2)


,Org_Name,Class
0,Gulf of California Marine Program,0
1,Philips Lighting,0
2,Department of Natural Resource Ecology and Man...,0
3,"Ministry of Agriculture and Forestry, Finland",1
4,National Science Foundation,1


In [7]:
# =============================================================================
# Validate required columns
# =============================================================================

required_columns = {TEXT_COLUMN, LABEL_COLUMN}
missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(
        f"The dataset is missing the following columns: {missing_columns}"
    )

# =============================================================================
# Preserve the original row order
# =============================================================================

df = df.copy()
df[ID_COLUMN] = np.arange(len(df))

# =============================================================================
# Clean text values
# =============================================================================

# Convert organization names to strings and remove leading/trailing whitespace.
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()

# =============================================================================
# Validate missing values
# =============================================================================

if df[TEXT_COLUMN].isna().any():
    raise ValueError(f"Missing values were found in {TEXT_COLUMN}.")

if df[LABEL_COLUMN].isna().any():
    raise ValueError(f"Missing values were found in {LABEL_COLUMN}.")

# =============================================================================
# Validate empty organization names
# =============================================================================

empty_text_count = df[TEXT_COLUMN].eq("").sum()

if empty_text_count > 0:
    raise ValueError(
        f"{empty_text_count} empty organization names were found."
    )

# =============================================================================
# Validate labels
# =============================================================================

df[LABEL_COLUMN] = pd.to_numeric(
    df[LABEL_COLUMN],
    errors="raise"
).astype(int)

valid_labels = {0, 1}
observed_labels = set(df[LABEL_COLUMN].unique())

if not observed_labels.issubset(valid_labels):
    raise ValueError(
        f"Unexpected labels were found: {observed_labels}. "
        "Only 0 and 1 are allowed."
    )

# =============================================================================
# Check duplicates
# =============================================================================

duplicate_name_count = df[TEXT_COLUMN].duplicated().sum()
duplicate_row_count = df[[TEXT_COLUMN, LABEL_COLUMN]].duplicated().sum()

# =============================================================================
# Display the data summary
# =============================================================================

print(f"Number of instances: {len(df):,}")
print(f"Missing organization names: {df[TEXT_COLUMN].isna().sum():,}")
print(f"Missing labels: {df[LABEL_COLUMN].isna().sum():,}")
print(f"Duplicate organization names: {duplicate_name_count:,}")
print(f"Duplicate text-label rows: {duplicate_row_count:,}")

print("\nClass distribution:")
print(df[LABEL_COLUMN].value_counts().sort_index())

print("\nClass proportions:")
print(
    df[LABEL_COLUMN]
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

Number of instances: 500
Missing organization names: 0
Missing labels: 0
Duplicate organization names: 0
Duplicate text-label rows: 0

Class distribution:
Class
0    245
1    255
Name: count, dtype: int64

Class proportions:
Class
0    0.49
1    0.51
Name: proportion, dtype: float64


In [8]:
# =============================================================================
# Inspect Multilingual Text
# =============================================================================


# Identify examples containing non-ASCII characters.
multilingual_examples = df[
    df[TEXT_COLUMN].apply(
        lambda text: any(ord(character) > 127 for character in text)
    )
][[ID_COLUMN, TEXT_COLUMN, LABEL_COLUMN]]

print(
    f"Instances containing at least one non-ASCII character: "
    f"{len(multilingual_examples):,}"
)

display(multilingual_examples.head(20))

Instances containing at least one non-ASCII character: 0


,Instance_ID,Org_Name,Class


## 5. Load the Tokenizer

In [9]:
# =============================================================================
# Load Tokenizer
# =============================================================================


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

print(f"Tokenizer loaded from: {MODEL_NAME}")
print(f"Tokenizer vocabulary size: {tokenizer.vocab_size:,}")
print(f"Maximum model length: {tokenizer.model_max_length}")

Tokenizer loaded from: google-bert/bert-base-multilingual-cased
Tokenizer vocabulary size: 119,547
Maximum model length: 512


In [10]:
# =============================================================================
# Examine Tokenization Examples
# =============================================================================


# Inspect how the selected tokenizer processes multilingual organization names.
sample_names = multilingual_examples[TEXT_COLUMN].head(10).tolist()

for organization_name in sample_names:
    tokens = tokenizer.tokenize(organization_name)

    print(f"Organization: {organization_name}")
    print(f"Tokens: {tokens}")
    print("-" * 80)

## 6. Define Tokenization and Evaluation Functions

In [11]:
# =============================================================================
# Tokenization Function
# =============================================================================


def tokenize_batch(batch):
    """Tokenize a batch of organization names."""

    return tokenizer(
        batch[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

In [12]:
# =============================================================================
# Metric Function for Trainer
# =============================================================================

def compute_trainer_metrics(eval_prediction):
    """Compute summary evaluation metrics during model training."""

    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    weighted_precision, weighted_recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1
    }

In [13]:
# =============================================================================
# Detailed Fold Metrics
# =============================================================================


def calculate_fold_metrics(
    true_labels,
    predicted_labels,
    fold_number
):
    """Calculate class-level and aggregate metrics for one fold."""

    report = classification_report(
        true_labels,
        predicted_labels,
        labels=[0, 1],
        target_names=["non-gov", "gov"],
        output_dict=True,
        zero_division=0
    )

    fold_metrics = {
        "fold": fold_number,
        "accuracy": accuracy_score(
            true_labels,
            predicted_labels
        ),

        # Class 0 metrics
        "class_0_precision": report["non-gov"]["precision"],
        "class_0_recall": report["non-gov"]["recall"],
        "class_0_f1": report["non-gov"]["f1-score"],
        "class_0_support": report["non-gov"]["support"],

        # Class 1 metrics
        "class_1_precision": report["gov"]["precision"],
        "class_1_recall": report["gov"]["recall"],
        "class_1_f1": report["gov"]["f1-score"],
        "class_1_support": report["gov"]["support"],

        # Macro-average metrics
        "macro_precision": report["macro avg"]["precision"],
        "macro_recall": report["macro avg"]["recall"],
        "macro_f1": report["macro avg"]["f1-score"],

        # Weighted-average metrics
        "weighted_precision": report["weighted avg"]["precision"],
        "weighted_recall": report["weighted avg"]["recall"],
        "weighted_f1": report["weighted avg"]["f1-score"]
    }

    return fold_metrics

In [14]:
# =============================================================================
# Softmax Function
# =============================================================================


def stable_softmax(logits):
    """Convert logits into probability scores using a stable softmax."""

    shifted_logits = logits - np.max(
        logits,
        axis=1,
        keepdims=True
    )

    exponentiated_logits = np.exp(shifted_logits)

    return exponentiated_logits / np.sum(
        exponentiated_logits,
        axis=1,
        keepdims=True
    )

In [15]:
# =============================================================================
# Model Initialization Function
# =============================================================================


def initialize_model():
    """Load a fresh sequence-classification model."""

    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=ID2LABEL,
        label2id=LABEL2ID
    )

## 7. Configure Stratified 10-Fold Cross-Validation

In [16]:
# =============================================================================
# Initialize Cross-Validation
# =============================================================================


stratified_kfold = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED
)

texts = df[TEXT_COLUMN].to_numpy()
labels = df[LABEL_COLUMN].to_numpy()

# Store fold-level metrics.
fold_metrics_list = []

# Store out-of-fold predictions for every instance.
out_of_fold_predictions = []

# Store fold-level confusion matrices.
confusion_matrices = []

## 8. Run Stratified 10-Fold Cross-Validation

In [17]:
# =============================================================================
# Main Cross-Validation Loop
# =============================================================================


for fold_number, (train_indices, validation_indices) in enumerate(
    stratified_kfold.split(texts, labels),
    start=1
):
    print("=" * 100)
    print(f"Starting fold {fold_number} of {N_SPLITS}")
    print("=" * 100)

    # -------------------------------------------------------------------------
    # Set a fold-specific random seed
    # -------------------------------------------------------------------------

    fold_seed = RANDOM_SEED + fold_number
    set_seed(fold_seed)

    # -------------------------------------------------------------------------
    # Create training and validation data frames
    # -------------------------------------------------------------------------

    train_df = df.iloc[train_indices].copy()
    validation_df = df.iloc[validation_indices].copy()

    print(f"Training instances: {len(train_df):,}")
    print(f"Validation instances: {len(validation_df):,}")

    print("\nTraining class distribution:")
    print(train_df[LABEL_COLUMN].value_counts().sort_index().to_dict())

    print("Validation class distribution:")
    print(validation_df[LABEL_COLUMN].value_counts().sort_index().to_dict())

    # -------------------------------------------------------------------------
    # Convert pandas data frames to Hugging Face datasets
    # -------------------------------------------------------------------------

    train_dataset = Dataset.from_pandas(
        train_df[[TEXT_COLUMN, LABEL_COLUMN]],
        preserve_index=False
    )

    validation_dataset = Dataset.from_pandas(
        validation_df[[TEXT_COLUMN, LABEL_COLUMN]],
        preserve_index=False
    )

    # Rename the target column because Trainer expects the name "labels".
    train_dataset = train_dataset.rename_column(
        LABEL_COLUMN,
        "labels"
    )

    validation_dataset = validation_dataset.rename_column(
        LABEL_COLUMN,
        "labels"
    )

    # -------------------------------------------------------------------------
    # Tokenize the datasets
    # -------------------------------------------------------------------------

    train_dataset = train_dataset.map(
        tokenize_batch,
        batched=True,
        desc=f"Tokenizing training data for fold {fold_number}"
    )

    validation_dataset = validation_dataset.map(
        tokenize_batch,
        batched=True,
        desc=f"Tokenizing validation data for fold {fold_number}"
    )

    # -------------------------------------------------------------------------
    # Create a dynamic padding collator
    # -------------------------------------------------------------------------

    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer
    )

    # -------------------------------------------------------------------------
    # Initialize a fresh pretrained model for the current fold
    # -------------------------------------------------------------------------

    model = initialize_model()

    fold_output_dir = CV_OUTPUT_DIR / f"fold_{fold_number}"

    # -------------------------------------------------------------------------
    # Define training arguments
    # -------------------------------------------------------------------------

    training_args = TrainingArguments(
        output_dir=str(fold_output_dir),

        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,

        # Evaluate and save the model after each epoch.
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        # Select the checkpoint with the highest macro F1.
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,

        # Retain only the best checkpoint and one recent checkpoint.
        save_total_limit=2,

        fp16=USE_FP16,

        # Set reproducibility-related seeds.
        seed=fold_seed,
        data_seed=fold_seed,

        # Disable external reporting services.
        report_to="none",

        # Keep all columns required by the model and metrics.
        remove_unused_columns=True
    )

    # -------------------------------------------------------------------------
    # Initialize Trainer
    # -------------------------------------------------------------------------

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_trainer_metrics
    )

    # -------------------------------------------------------------------------
    # Train the model
    # -------------------------------------------------------------------------

    trainer.train()

    # -------------------------------------------------------------------------
    # Predict the validation fold
    # -------------------------------------------------------------------------

    prediction_output = trainer.predict(
        validation_dataset
    )

    fold_logits = prediction_output.predictions
    fold_probabilities = stable_softmax(fold_logits)
    fold_predictions = np.argmax(
        fold_probabilities,
        axis=1
    )

    true_labels = validation_df[LABEL_COLUMN].to_numpy()

    # -------------------------------------------------------------------------
    # Calculate fold-level metrics
    # -------------------------------------------------------------------------

    fold_metrics = calculate_fold_metrics(
        true_labels=true_labels,
        predicted_labels=fold_predictions,
        fold_number=fold_number
    )

    
    
    fold_metrics["best_checkpoint"] = (
        trainer.state.best_model_checkpoint
        if KEEP_FOLD_CHECKPOINTS
    else "Deleted after fold evaluation"
    )

    fold_metrics["best_validation_metric"] = (
        trainer.state.best_metric
    )

    fold_metrics_list.append(fold_metrics)

    # -------------------------------------------------------------------------
    # Calculate and store the confusion matrix
    # -------------------------------------------------------------------------

    fold_confusion_matrix = confusion_matrix(
        true_labels,
        fold_predictions,
        labels=[0, 1]
    )

    confusion_matrices.append(
        {
            "fold": fold_number,
            "true_0_pred_0": int(fold_confusion_matrix[0, 0]),
            "true_0_pred_1": int(fold_confusion_matrix[0, 1]),
            "true_1_pred_0": int(fold_confusion_matrix[1, 0]),
            "true_1_pred_1": int(fold_confusion_matrix[1, 1])
        }
    )

    # -------------------------------------------------------------------------
    # Store instance-level out-of-fold predictions
    # -------------------------------------------------------------------------

    for local_position, original_index in enumerate(
        validation_indices
    ):
        original_row = df.iloc[original_index]

        out_of_fold_predictions.append(
            {
                ID_COLUMN: int(original_row[ID_COLUMN]),
                TEXT_COLUMN: original_row[TEXT_COLUMN],
                "True_Class": int(true_labels[local_position]),
                "True_Label": ID2LABEL[
                    int(true_labels[local_position])
                ],
                "Predicted_Class": int(
                    fold_predictions[local_position]
                ),
                "Predicted_Label": ID2LABEL[
                    int(fold_predictions[local_position])
                ],
                "Probability_Class_0_non_gov": float(
                    fold_probabilities[local_position, 0]
                ),
                "Probability_Class_1_gov": float(
                    fold_probabilities[local_position, 1]
                ),
                "Prediction_Confidence": float(
                    np.max(fold_probabilities[local_position])
                ),
                "Correct_Prediction": bool(
                    fold_predictions[local_position]
                    == true_labels[local_position]
                ),
                "Fold": fold_number,
                "Model_Choice": MODEL_CHOICE,
                "Model_Name": MODEL_NAME
            }
        )

    # -------------------------------------------------------------------------
    # Display current fold results
    # -------------------------------------------------------------------------

    print(f"\nFold {fold_number} results:")

    for metric_name, metric_value in fold_metrics.items():
        if isinstance(metric_value, (float, np.floating)):
            print(f"{metric_name}: {metric_value:.4f}")

    print("\nConfusion matrix:")
    print(fold_confusion_matrix)

    # -------------------------------------------------------------------------
    # Save the checkpoint path before deleting the Trainer object
    # -------------------------------------------------------------------------

    best_checkpoint_path = trainer.state.best_model_checkpoint

    # -------------------------------------------------------------------------
    # Release memory before starting the next fold
    # -------------------------------------------------------------------------

    del trainer
    del model
    del train_dataset
    del validation_dataset
    del prediction_output
    del fold_logits
    del fold_probabilities

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # -------------------------------------------------------------------------
    # Delete fold checkpoints to conserve disk space
    # -------------------------------------------------------------------------

    if not KEEP_FOLD_CHECKPOINTS and fold_output_dir.exists():
        shutil.rmtree(fold_output_dir)

        print(
            f"Deleted checkpoints for fold {fold_number}: "
            f"{fold_output_dir}"
        )

Starting fold 1 of 10
Training instances: 450
Validation instances: 50

Training class distribution:
{0: 220, 1: 230}
Validation class distribution:
{0: 25, 1: 25}


Tokenizing training data for fold 1:   0%|          | 0/450 [00:00<?, ? examples/s]

Tokenizing validation data for fold 1:   0%|          | 0/50 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
1,0.601365,0.416063,0.800000,0.834225,0.800000,0.794745,0.834225,0.800000,0.794745
2,0.354324,0.717184,0.740000,0.797619,0.740000,0.726776,0.797619,0.740000,0.726776
3,0.197577,0.384877,0.920000,0.920000,0.920000,0.920000,0.920000,0.920000,0.920000
4,0.181884,0.539715,0.860000,0.865260,0.860000,0.859494,0.865260,0.860000,0.859494
5,0.073037,0.523626,0.900000,0.900641,0.900000,0.899960,0.900641,0.900000,0.899960


... (111 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte


Fold 10 results:
accuracy: 0.8400
class_0_precision: 0.8636
class_0_recall: 0.7917
class_0_f1: 0.8261
class_0_support: 24.0000
class_1_precision: 0.8214
class_1_recall: 0.8846
class_1_f1: 0.8519
class_1_support: 26.0000
macro_precision: 0.8425
macro_recall: 0.8381
macro_f1: 0.8390
weighted_precision: 0.8417
weighted_recall: 0.8400
weighted_f1: 0.8395
best_validation_metric: 0.8390

Confusion matrix:
[[19  5]
 [ 3 23]]
Deleted checkpoints for fold 10: organization_classification_results/cross_validation/fold_10


## 9. Save Fold-Level Evaluation Results

In [18]:
# =============================================================================
# Fold Results
# =============================================================================


fold_metrics_df = pd.DataFrame(
    fold_metrics_list
).sort_values("fold")

fold_metrics_file = (
    RESULTS_DIR
    / f"{MODEL_CHOICE}_10fold_metrics.csv"
)

fold_metrics_df.to_csv(
    fold_metrics_file,
    index=False,
    encoding="utf-8-sig"
)

display(fold_metrics_df)

print(f"Fold-level metrics saved to: {fold_metrics_file}")

,fold,accuracy,class_0_precision,class_0_recall,class_0_f1,class_0_support,class_1_precision,class_1_recall,class_1_f1,class_1_support,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,best_checkpoint,best_validation_metric
0,1,0.92,0.920000,0.920000,0.920000,25.0,0.920000,0.920000,0.920000,25.0,0.920000,0.920000,0.920000,0.920000,0.92,0.920000,Deleted after fold evaluation,0.920000
1,2,0.84,0.793103,0.920000,0.851852,25.0,0.904762,0.760000,0.826087,25.0,0.848933,0.840000,0.838969,0.848933,0.84,0.838969,Deleted after fold evaluation,0.838969
2,3,0.88,0.880000,0.880000,0.880000,25.0,0.880000,0.880000,0.880000,25.0,0.880000,0.880000,0.880000,0.880000,0.88,0.880000,Deleted after fold evaluation,0.880000
3,4,0.88,0.851852,0.920000,0.884615,25.0,0.913043,0.840000,0.875000,25.0,0.882448,0.880000,0.879808,0.882448,0.88,0.879808,Deleted after fold evaluation,0.879808
4,5,0.92,0.920000,0.920000,0.920000,25.0,0.920000,0.920000,0.920000,25.0,0.920000,0.920000,0.920000,0.920000,0.92,0.920000,Deleted after fold evaluation,0.920000
5,6,0.80,0.769231,0.833333,0.800000,24.0,0.833333,0.769231,0.800000,26.0,0.801282,0.801282,0.800000,0.802564,0.80,0.800000,Deleted after fold evaluation,0.800000
6,7,0.80,0.818182,0.750000,0.782609,24.0,0.785714,0.846154,0.814815,26.0,0.801948,0.798077,0.798712,0.801299,0.80,0.799356,Deleted after fold evaluation,0.798712
7,8,0.90,0.851852,0.958333,0.901961,24.0,0.956522,0.846154,0.897959,26.0,0.904187,0.902244,0.899960,0.906280,0.90,0.899880,Deleted after fold evaluation,0.899960
8,9,0.92,0.916667,0.916667,0.916667,24.0,0.923077,0.923077,0.923077,26.0,0.919872,0.919872,0.919872,0.920000,0.92,0.920000,Deleted after fold evaluation,0.919872
9,10,0.84,0.863636,0.791667,0.826087,24.0,0.821429,0.884615,0.851852,26.0,0.842532,0.838141,0.838969,0.841688,0.84,0.839485,Deleted after fold evaluation,0.838969


Fold-level metrics saved to: organization_classification_results/evaluation_results/bert_10fold_metrics.csv


## 10. Calculate the Mean and Standard Deviation Across Folds

In [19]:
# =============================================================================
# Aggregate Metrics
# =============================================================================


# Select only numeric evaluation metrics.
metric_columns = [
    "accuracy",

    "class_0_precision",
    "class_0_recall",
    "class_0_f1",

    "class_1_precision",
    "class_1_recall",
    "class_1_f1",

    "macro_precision",
    "macro_recall",
    "macro_f1",

    "weighted_precision",
    "weighted_recall",
    "weighted_f1"
]

# Calculate the mean and sample standard deviation across the 10 folds.
aggregate_metrics_df = pd.DataFrame(
    {
        "Metric": metric_columns,
        "Mean": [
            fold_metrics_df[column].mean()
            for column in metric_columns
        ],
        "Standard_Deviation": [
            fold_metrics_df[column].std(ddof=1)
            for column in metric_columns
        ],
        "Minimum": [
            fold_metrics_df[column].min()
            for column in metric_columns
        ],
        "Maximum": [
            fold_metrics_df[column].max()
            for column in metric_columns
        ]
    }
)

aggregate_metrics_df[
    ["Mean", "Standard_Deviation", "Minimum", "Maximum"]
] = aggregate_metrics_df[
    ["Mean", "Standard_Deviation", "Minimum", "Maximum"]
].round(4)

aggregate_metrics_file = (
    RESULTS_DIR
    / f"{MODEL_CHOICE}_10fold_aggregate_metrics.csv"
)

aggregate_metrics_df.to_csv(
    aggregate_metrics_file,
    index=False,
    encoding="utf-8-sig"
)

display(aggregate_metrics_df)

print(
    f"Aggregate metrics saved to: "
    f"{aggregate_metrics_file}"
)

,Metric,Mean,Standard_Deviation,Minimum,Maximum
0,accuracy,0.8700,0.0474,0.8000,0.9200
1,class_0_precision,0.8585,0.0531,0.7692,0.9200
2,class_0_recall,0.8810,0.0673,0.7500,0.9583
3,class_0_f1,0.8684,0.0509,0.7826,0.9200
4,class_1_precision,0.8858,0.0545,0.7857,0.9565
5,class_1_recall,0.8589,0.0589,0.7600,0.9231
6,class_1_f1,0.8709,0.0458,0.8000,0.9231
7,macro_precision,0.8721,0.0464,0.8013,0.9200
8,macro_recall,0.8700,0.0478,0.7981,0.9200
9,macro_f1,0.8696,0.0477,0.7987,0.9200


Aggregate metrics saved to: organization_classification_results/evaluation_results/bert_10fold_aggregate_metrics.csv


## 11. Save Fold-Level Confusion Matrices

In [20]:
# =============================================================================
# Confusion Matrices
# =============================================================================


confusion_matrices_df = pd.DataFrame(
    confusion_matrices
).sort_values("fold")

confusion_matrix_file = (
    RESULTS_DIR
    / f"{MODEL_CHOICE}_10fold_confusion_matrices.csv"
)

confusion_matrices_df.to_csv(
    confusion_matrix_file,
    index=False,
    encoding="utf-8-sig"
)

display(confusion_matrices_df)

print(
    f"Confusion matrices saved to: "
    f"{confusion_matrix_file}"
)

,fold,true_0_pred_0,true_0_pred_1,true_1_pred_0,true_1_pred_1
0,1,23,2,2,23
1,2,23,2,6,19
2,3,22,3,3,22
3,4,23,2,4,21
4,5,23,2,2,23
5,6,20,4,6,20
6,7,18,6,4,22
7,8,23,1,4,22
8,9,22,2,2,24
9,10,19,5,3,23


Confusion matrices saved to: organization_classification_results/evaluation_results/bert_10fold_confusion_matrices.csv


## 12. Save Instance-Level Out-of-Fold Predictions

In [21]:
# =============================================================================
# Save All Predictions and Probabilities
# =============================================================================


out_of_fold_predictions_df = pd.DataFrame(
    out_of_fold_predictions
).sort_values(ID_COLUMN)

# Verify that every original instance received exactly one out-of-fold prediction.
if len(out_of_fold_predictions_df) != len(df):
    raise RuntimeError(
        "The number of out-of-fold predictions does not match "
        "the number of original instances."
    )

if out_of_fold_predictions_df[ID_COLUMN].duplicated().any():
    raise RuntimeError(
        "At least one instance received more than one "
        "out-of-fold prediction."
    )

prediction_file = (
    RESULTS_DIR
    / f"{MODEL_CHOICE}_10fold_instance_predictions.csv"
)

out_of_fold_predictions_df.to_csv(
    prediction_file,
    index=False,
    encoding="utf-8-sig"
)

display(out_of_fold_predictions_df.head(20))

print(
    f"Out-of-fold predictions saved to: "
    f"{prediction_file}"
)

,Instance_ID,Org_Name,True_Class,True_Label,Predicted_Class,Predicted_Label,Probability_Class_0_non_gov,Probability_Class_1_gov,Prediction_Confidence,Correct_Prediction,Fold,Model_Choice,Model_Name
0,0,Gulf of California Marine Program,0,non-gov,1,gov,0.007592,0.992408,0.992408,False,1,bert,google-bert/bert-base-multilingual-cased
1,1,Philips Lighting,0,non-gov,0,non-gov,0.995669,0.004332,0.995669,True,1,bert,google-bert/bert-base-multilingual-cased
300,2,Department of Natural Resource Ecology and Man...,0,non-gov,0,non-gov,0.991700,0.008300,0.991700,True,7,bert,google-bert/bert-base-multilingual-cased
301,3,"Ministry of Agriculture and Forestry, Finland",1,gov,1,gov,0.008762,0.991238,0.991238,True,7,bert,google-bert/bert-base-multilingual-cased
450,4,National Science Foundation,1,gov,1,gov,0.006654,0.993346,0.993346,True,10,bert,google-bert/bert-base-multilingual-cased
150,5,AGFACE,1,gov,1,gov,0.007786,0.992214,0.992214,True,4,bert,google-bert/bert-base-multilingual-cased
100,6,"Forest Service, Savannah River",1,gov,1,gov,0.002627,0.997373,0.997373,True,3,bert,google-bert/bert-base-multilingual-cased
302,7,NZAID,1,gov,1,gov,0.088229,0.911771,0.911771,True,7,bert,google-bert/bert-base-multilingual-cased
151,8,Atmospheric Environment Service,1,gov,1,gov,0.002364,0.997636,0.997636,True,4,bert,google-bert/bert-base-multilingual-cased
400,9,University of Aberdeen through a Systems Biolo...,0,non-gov,0,non-gov,0.995523,0.004477,0.995523,True,9,bert,google-bert/bert-base-multilingual-cased


Out-of-fold predictions saved to: organization_classification_results/evaluation_results/bert_10fold_instance_predictions.csv


## 13. Calculate Overall Out-of-Fold Performance

In [22]:
# =============================================================================
# Pooled Out-of-Fold Evaluation
# =============================================================================



oof_true_labels = out_of_fold_predictions_df[
    "True_Class"
].to_numpy()

oof_predicted_labels = out_of_fold_predictions_df[
    "Predicted_Class"
].to_numpy()

oof_report = classification_report(
    oof_true_labels,
    oof_predicted_labels,
    labels=[0, 1],
    target_names=["non-gov", "gov"],
    digits=4,
    zero_division=0
)

oof_confusion_matrix = confusion_matrix(
    oof_true_labels,
    oof_predicted_labels,
    labels=[0, 1]
)

print("Overall out-of-fold classification report:")
print(oof_report)

print("Overall out-of-fold confusion matrix:")
print(oof_confusion_matrix)

Overall out-of-fold classification report:
              precision    recall  f1-score   support

     non-gov     0.8571    0.8816    0.8692       245
         gov     0.8831    0.8588    0.8708       255

    accuracy                         0.8700       500
   macro avg     0.8701    0.8702    0.8700       500
weighted avg     0.8704    0.8700    0.8700       500

Overall out-of-fold confusion matrix:
[[216  29]
 [ 36 219]]


## 14. Save Misclassified Instances

In [23]:
# =============================================================================
# Misclassification Analysis
# =============================================================================


misclassified_df = out_of_fold_predictions_df[
    ~out_of_fold_predictions_df["Correct_Prediction"]
].copy()

# Sort errors by prediction confidence in descending order.
misclassified_df = misclassified_df.sort_values(
    "Prediction_Confidence",
    ascending=False
)

misclassified_file = (
    RESULTS_DIR
    / f"{MODEL_CHOICE}_misclassified_instances.csv"
)

misclassified_df.to_csv(
    misclassified_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"Number of misclassified instances: {len(misclassified_df):,}")

display(misclassified_df.head(30))

print(
    f"Misclassified instances saved to: "
    f"{misclassified_file}"
)

Number of misclassified instances: 65


,Instance_ID,Org_Name,True_Class,True_Label,Predicted_Class,Predicted_Label,Probability_Class_0_non_gov,Probability_Class_1_gov,Prediction_Confidence,Correct_Prediction,Fold,Model_Choice,Model_Name
77,285,Foundation for the National Institutes of Health,0,non-gov,1,gov,0.001741,0.998259,0.998259,False,2,bert,google-bert/bert-base-multilingual-cased
499,499,Canadian Graduate Scholarships Program,1,gov,0,non-gov,0.998183,0.001817,0.998183,False,10,bert,google-bert/bert-base-multilingual-cased
132,307,Conservation International's southern Africa's...,0,non-gov,1,gov,0.001879,0.998121,0.998121,False,3,bert,google-bert/bert-base-multilingual-cased
120,160,Canadian Centennial Scholarship Fund,0,non-gov,1,gov,0.002243,0.997757,0.997757,False,3,bert,google-bert/bert-base-multilingual-cased
495,444,Direccion General de Asuntos de Personal Acade...,0,non-gov,1,gov,0.002621,0.997379,0.997379,False,10,bert,google-bert/bert-base-multilingual-cased
231,309,"Fish and Wildlife Research Institute, St Peter...",1,gov,0,non-gov,0.997264,0.002736,0.997264,False,5,bert,google-bert/bert-base-multilingual-cased
154,46,Southwest Fisheries Science Center,1,gov,0,non-gov,0.997172,0.002828,0.997172,False,4,bert,google-bert/bert-base-multilingual-cased
91,357,PRONEX,1,gov,0,non-gov,0.997088,0.002912,0.997088,False,2,bert,google-bert/bert-base-multilingual-cased
219,155,PRCF,0,non-gov,1,gov,0.002924,0.997077,0.997077,False,5,bert,google-bert/bert-base-multilingual-cased
158,83,Royal Botanic Garden Edinburgh,1,gov,0,non-gov,0.996809,0.003191,0.996809,False,4,bert,google-bert/bert-base-multilingual-cased


Misclassified instances saved to: organization_classification_results/evaluation_results/bert_misclassified_instances.csv


## 15. Retrain the Final Model on the Full Dataset

In [24]:
# =============================================================================
# Prepare the Full Dataset
# =============================================================================


# Create a Hugging Face dataset using all 500 labeled instances.
full_dataset = Dataset.from_pandas(
    df[[TEXT_COLUMN, LABEL_COLUMN]],
    preserve_index=False
)

full_dataset = full_dataset.rename_column(
    LABEL_COLUMN,
    "labels"
)

full_dataset = full_dataset.map(
    tokenize_batch,
    batched=True,
    desc="Tokenizing the full dataset"
)

full_data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print(full_dataset)

Tokenizing the full dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['Org_Name', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 500
})


In [25]:
# =============================================================================
# Train the Final Deployment Model
# =============================================================================


# Set the original seed before final model training.
set_seed(RANDOM_SEED)

# Initialize a fresh pretrained model.
final_model = initialize_model()

final_training_output_dir = (
    FINAL_MODEL_DIR
    / "training_checkpoints"
)

final_training_args = TrainingArguments(
    output_dir=str(final_training_output_dir),

    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    # No evaluation set is used because all labeled data are used for training.
    eval_strategy="no",

    # Do not retain intermediate checkpoints for the final full-data model.
    save_strategy="no",

    logging_strategy="epoch",

    fp16=USE_FP16,

    seed=RANDOM_SEED,
    data_seed=RANDOM_SEED,

    report_to="none",
    remove_unused_columns=True
)

final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=full_dataset,
    processing_class=tokenizer,
    data_collator=full_data_collator
)

final_training_result = final_trainer.train()

print("Final full-data model training completed.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

Step,Training Loss
63,0.580526
126,0.322991
189,0.174169
252,0.108275
315,0.082048


Final full-data model training completed.


## 16. Save the Final Model and Tokenizer for Deployment

In [26]:
# =============================================================================
# Save Deployment Artifacts
# =============================================================================


deployment_model_dir = (
    FINAL_MODEL_DIR
    / f"{MODEL_CHOICE}_deployment_model"
)

deployment_model_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Save the model, configuration, and tokenizer.
final_trainer.save_model(
    str(deployment_model_dir)
)

tokenizer.save_pretrained(
    str(deployment_model_dir)
)

# Save experiment metadata.
experiment_metadata = {
    "model_choice": MODEL_CHOICE,
    "pretrained_model_name": MODEL_NAME,
    "text_column": TEXT_COLUMN,
    "label_column": LABEL_COLUMN,
    "number_of_training_instances": int(len(df)),
    "number_of_classes": 2,
    "id2label": {
        str(key): value
        for key, value in ID2LABEL.items()
    },
    "label2id": LABEL2ID,
    "max_length": MAX_LENGTH,
    "learning_rate": LEARNING_RATE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "number_of_epochs": NUM_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "random_seed": RANDOM_SEED,
    "cross_validation_folds": N_SPLITS,
    "transformers_version": transformers.__version__,
    "torch_version": torch.__version__,
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
    "python_version": sys.version
}

metadata_file = (
    deployment_model_dir
    / "experiment_metadata.json"
)

with open(
    metadata_file,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        experiment_metadata,
        file,
        ensure_ascii=False,
        indent=4
    )

print(
    f"Final deployment model saved to: "
    f"{deployment_model_dir}"
)

print(
    f"Experiment metadata saved to: "
    f"{metadata_file}"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final deployment model saved to: organization_classification_results/final_model/bert_deployment_model
Experiment metadata saved to: organization_classification_results/final_model/bert_deployment_model/experiment_metadata.json


## 17. Test the Saved Deployment Model

In [27]:
# =============================================================================
# Reload the Saved Model
# =============================================================================


# Reload the model and tokenizer from the saved deployment directory.
loaded_tokenizer = AutoTokenizer.from_pretrained(
    str(deployment_model_dir)
)

loaded_model = AutoModelForSequenceClassification.from_pretrained(
    str(deployment_model_dir)
)

loaded_model.eval()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

loaded_model.to(device)

print(f"Deployment model loaded on: {device}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Deployment model loaded on: cuda


In [28]:
# =============================================================================
# Define a Prediction Function
# =============================================================================


def predict_organization(
    organization_name,
    model,
    tokenizer_object,
    model_device
):
    """Predict the class and probability scores for one organization name."""

    encoded_input = tokenizer_object(
        organization_name,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True
    )

    encoded_input = {
        key: value.to(model_device)
        for key, value in encoded_input.items()
    }

    with torch.no_grad():
        model_output = model(**encoded_input)

    logits = model_output.logits.detach().cpu().numpy()
    probabilities = stable_softmax(logits)[0]

    predicted_class = int(np.argmax(probabilities))

    return {
        "Org_Name": organization_name,
        "Predicted_Class": predicted_class,
        "Predicted_Label": ID2LABEL[predicted_class],
        "Probability_Class_0_non_gov": float(probabilities[0]),
        "Probability_Class_1_gov": float(probabilities[1]),
        "Prediction_Confidence": float(
            probabilities[predicted_class]
        )
    }

In [29]:
# =============================================================================
# Test Prediction
# =============================================================================


test_organization = "United States Department of Energy"

test_prediction = predict_organization(
    organization_name=test_organization,
    model=loaded_model,
    tokenizer_object=loaded_tokenizer,
    model_device=device
)

print(json.dumps(
    test_prediction,
    ensure_ascii=False,
    indent=4
))

{
    "Org_Name": "United States Department of Energy",
    "Predicted_Class": 1,
    "Predicted_Label": "gov",
    "Probability_Class_0_non_gov": 0.0015100963646546006,
    "Probability_Class_1_gov": 0.9984898567199707,
    "Prediction_Confidence": 0.9984898567199707
}


## 18. Batch Prediction for New Organization Names

In [30]:
# =============================================================================
# Batch Prediction Function
# =============================================================================


def predict_organization_batch(
    organization_names,
    model,
    tokenizer_object,
    model_device,
    batch_size=32
):
    """Predict classes and probabilities for multiple organization names."""

    prediction_records = []

    for batch_start in range(
        0,
        len(organization_names),
        batch_size
    ):
        batch_names = organization_names[
            batch_start:batch_start + batch_size
        ]

        encoded_batch = tokenizer_object(
            batch_names,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True
        )

        encoded_batch = {
            key: value.to(model_device)
            for key, value in encoded_batch.items()
        }

        with torch.no_grad():
            model_output = model(**encoded_batch)

        logits = model_output.logits.detach().cpu().numpy()
        probabilities = stable_softmax(logits)
        predicted_classes = np.argmax(
            probabilities,
            axis=1
        )

        for index, organization_name in enumerate(batch_names):
            predicted_class = int(
                predicted_classes[index]
            )

            prediction_records.append(
                {
                    "Org_Name": organization_name,
                    "Predicted_Class": predicted_class,
                    "Predicted_Label": ID2LABEL[
                        predicted_class
                    ],
                    "Probability_Class_0_non_gov": float(
                        probabilities[index, 0]
                    ),
                    "Probability_Class_1_gov": float(
                        probabilities[index, 1]
                    ),
                    "Prediction_Confidence": float(
                        probabilities[
                            index,
                            predicted_class
                        ]
                    )
                }
            )

    return pd.DataFrame(prediction_records)

In [31]:
# =============================================================================
# Batch Prediction Example
# =============================================================================


new_organization_names = [
    "National Science Foundation",
    "University of Oxford",
    "Ministério da Ciência, Tecnologia e Inovação",
    "Torsten Söderbergs Stiftelse"
]

batch_prediction_df = predict_organization_batch(
    organization_names=new_organization_names,
    model=loaded_model,
    tokenizer_object=loaded_tokenizer,
    model_device=device
)

display(batch_prediction_df)

batch_prediction_df.to_csv(
    RESULTS_DIR / f"{MODEL_CHOICE}_new_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

,Org_Name,Predicted_Class,Predicted_Label,Probability_Class_0_non_gov,Probability_Class_1_gov,Prediction_Confidence
0,National Science Foundation,1,gov,0.001394,0.998607,0.998607
1,University of Oxford,0,non-gov,0.998301,0.001699,0.998301
2,"Ministério da Ciência, Tecnologia e Inovação",1,gov,0.001631,0.998370,0.998370
3,Torsten Söderbergs Stiftelse,0,non-gov,0.998061,0.001939,0.998061


## 19. Export an Experiment Summary

In [32]:
# =============================================================================
# Save a Human-Readable Summary
# =============================================================================


summary_lines = [
    "Organization Name Binary Classification Experiment",
    "=" * 60,
    f"Model choice: {MODEL_CHOICE}",
    f"Pretrained model: {MODEL_NAME}",
    f"Number of instances: {len(df)}",
    f"Number of folds: {N_SPLITS}",
    f"Class 0 count: {(df[LABEL_COLUMN] == 0).sum()}",
    f"Class 1 count: {(df[LABEL_COLUMN] == 1).sum()}",
    "",
    "Mean 10-fold evaluation results",
    "-" * 60
]

for _, metric_row in aggregate_metrics_df.iterrows():
    summary_lines.append(
        f"{metric_row['Metric']}: "
        f"{metric_row['Mean']:.4f} "
        f"(SD = {metric_row['Standard_Deviation']:.4f})"
    )

summary_text = "\n".join(summary_lines)

summary_file = (
    RESULTS_DIR
    / f"{MODEL_CHOICE}_experiment_summary.txt"
)

with open(
    summary_file,
    "w",
    encoding="utf-8"
) as file:
    file.write(summary_text)

print(summary_text)
print(f"\nSummary saved to: {summary_file}")

Organization Name Binary Classification Experiment
Model choice: bert
Pretrained model: google-bert/bert-base-multilingual-cased
Number of instances: 500
Number of folds: 10
Class 0 count: 245
Class 1 count: 255

Mean 10-fold evaluation results
------------------------------------------------------------
accuracy: 0.8700 (SD = 0.0474)
class_0_precision: 0.8585 (SD = 0.0531)
class_0_recall: 0.8810 (SD = 0.0673)
class_0_f1: 0.8684 (SD = 0.0509)
class_1_precision: 0.8858 (SD = 0.0545)
class_1_recall: 0.8589 (SD = 0.0589)
class_1_f1: 0.8709 (SD = 0.0458)
macro_precision: 0.8721 (SD = 0.0464)
macro_recall: 0.8700 (SD = 0.0478)
macro_f1: 0.8696 (SD = 0.0477)
weighted_precision: 0.8723 (SD = 0.0466)
weighted_recall: 0.8700 (SD = 0.0474)
weighted_f1: 0.8697 (SD = 0.0476)

Summary saved to: organization_classification_results/evaluation_results/bert_experiment_summary.txt


## 19. Cleaning

In [ ]:
# =============================================================================
# Clean memory
# =============================================================================

import gc
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()